In [11]:
# --- CELL 1: SQLITE DATABASE SETUP & AUTOMATED YTD FETCH ---
import os
import sqlite3
import pandas as pd

DB_NAME = "payroll_ledger.db"
CSV_BACKUP_NAME = "payroll_history_backup.csv"
EMPLOYEE_ID = "EE-001"  # Identifier for your single employee

# 1. Initialize SQLite Database and create the Ledger Table
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS payroll_ledger (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id TEXT,
    pay_period_ending TEXT,
    gross_wages REAL,
    pre_tax_401k REAL,
    ee_ss REAL, ee_medicare REAL, ee_fed_tax REAL, ee_nm_tax REAL, ee_wc_fee REAL,
    net_pay REAL,
    er_ss REAL, er_medicare REAL, er_futa REAL, er_nm_sui REAL, er_wc_fee REAL,
    total_company_cost REAL,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

# 2. Automatically compute historical YTD Gross for this Employee ID
cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
result = cursor.fetchone()[0]
ytd_gross_before_current_pay = float(result) if result is not None else 0.00

conn.close()
print(f"Database tracking connected.")
print(f"Automated YTD Gross detected for {EMPLOYEE_ID}: ${ytd_gross_before_current_pay:,.2f}")


Database tracking connected.
Automated YTD Gross detected for EE-001: $10,015.38


In [12]:
# --- CELL 2: INPUT CONFIGURATION & 2026 REGULATORY STATS ---

# Current Pay Period Information
pay_period_date = "2026-06-15"          # Update this date string for every payroll run
base_gross_per_period = 5000.00         # 
taxable_health_reimbursement = 200.00   # Taxable reimbursement for S-Corp Owner
pay_periods_per_year = 12

# Pre-Tax 401k calculations
pre_tax_401k_rate = 0.05               
pre_tax_401k_per_period = base_gross_per_period * pre_tax_401k_rate
current_period_gross = base_gross_per_period + taxable_health_reimbursement

# Statutory Limits & Configured Rates
SS_WAGE_BASE_2026 = 184500.00      
FUTA_WAGE_BASE_2026 = 7000.00      
NM_SUI_WAGE_BASE_2026 = 34800.00   

SS_RATE = 0.062
MEDICARE_RATE = 0.0145
FUTA_NET_RATE = 0.006              
NM_SUI_EMPLOYER_RATE = 0.034       # Configured to your specific 3.4% rate
NM_WC_FEE_PER_PAY = (2.55 * 4) / pay_periods_per_year # $2.55/quarter normalized

# Progressive Income Tax Tables
FED_BRACKETS_2026 = [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)]
NM_BRACKETS_2026 = [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]


In [13]:
# --- CELL 3: RUN MATH CALCULATION ENGINES ---

def calculate_capped_tax(current_gross, ytd_prior, wage_cap, tax_rate):
    if ytd_prior >= wage_cap:
        return 0.00
    wages_subject_to_tax = min(current_gross, wage_cap - ytd_prior)
    return wages_subject_to_tax * tax_rate

def calculate_progressive_withholding(taxable_gross_period, periods, brackets):
    annual_taxable = taxable_gross_period * periods
    tax_owed = 0.0
    previous_limit = 0.0
    for limit, rate in brackets:
        if annual_taxable > limit:
            tax_owed += (limit - previous_limit) * rate
            previous_limit = limit
        else:
            tax_owed += (annual_taxable - previous_limit) * rate
            break
    return tax_owed / periods

# Split subject wages base parameters
fica_subject_gross = current_period_gross
income_tax_subject_gross = current_period_gross - pre_tax_401k_per_period

# Employee Liabilities
ee_social_security = calculate_capped_tax(fica_subject_gross, ytd_gross_before_current_pay, SS_WAGE_BASE_2026, SS_RATE)
ee_medicare = fica_subject_gross * MEDICARE_RATE
ee_fed_withholding = calculate_progressive_withholding(income_tax_subject_gross, pay_periods_per_year, FED_BRACKETS_2026)
ee_nm_withholding = calculate_progressive_withholding(income_tax_subject_gross, pay_periods_per_year, NM_BRACKETS_2026)

total_ee_taxes = ee_social_security + ee_medicare + ee_fed_withholding + ee_nm_withholding + NM_WC_FEE_PER_PAY
net_take_home_pay = current_period_gross - pre_tax_401k_per_period - total_ee_taxes

# Employer Liabilities
er_social_security = ee_social_security
er_medicare = ee_medicare
er_futa = calculate_capped_tax(fica_subject_gross, ytd_gross_before_current_pay, FUTA_WAGE_BASE_2026, FUTA_NET_RATE)
er_nm_sui = calculate_capped_tax(fica_subject_gross, ytd_gross_before_current_pay, NM_SUI_WAGE_BASE_2026, NM_SUI_EMPLOYER_RATE)
total_er_taxes = er_social_security + er_medicare + er_futa + er_nm_sui + NM_WC_FEE_PER_PAY
total_company_cash_outflow = current_period_gross + total_er_taxes


In [14]:
# --- CELL 4: PERSIST PAYROLL ENTRY TO SQLITE & EXPORT CSV ---

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# 1. Insert structured period calculations into SQLite
cursor.execute("""
INSERT INTO payroll_ledger (
    employee_id, pay_period_ending, gross_wages, pre_tax_401k,
    ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay,
    er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", (
    EMPLOYEE_ID, pay_period_date, current_period_gross, pre_tax_401k_per_period,
    ee_social_security, ee_medicare, ee_fed_withholding, ee_nm_withholding, NM_WC_FEE_PER_PAY, net_take_home_pay,
    er_social_security, er_medicare, er_futa, er_nm_sui, NM_WC_FEE_PER_PAY, total_company_cash_outflow
))
conn.commit()

# 2. Extract entire operational history to Pandas DataFrame and back up to CSV
df_history = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
df_history.to_csv(CSV_BACKUP_NAME, index=False)

conn.close()
print(f"Success: Record inserted into SQLite database '{DB_NAME}'.")
print(f"Success: Database verified and backed up to flat ledger CSV '{CSV_BACKUP_NAME}'.")



Success: Record inserted into SQLite database 'payroll_ledger.db'.
Success: Database verified and backed up to flat ledger CSV 'payroll_history_backup.csv'.


In [15]:
# --- CELL 5: COMPREHENSIVE CURRENT PER-CHECK & YTD VISUAL STUB ---

print("=" * 60)
print(f"      PAYROLL OVERVIEW FOR: {EMPLOYEE_ID} | PERIOD: {pay_period_date}   ")
print("=" * 60)
print(f"Current Period Gross Wages:            ${current_period_gross:12,.2f}")
print(f"Current Pre-Tax 401(k) Allocation:     ${pre_tax_401k_per_period:12,.2f}")
print(f"EMPLOYEE NET TAKE-HOME PAY:            ${net_take_home_pay:12,.2f}")
print("-" * 60)
print(f"Employer Matching FICA Social Security: ${er_social_security:12,.2f}")
print(f"Employer State Unemployment (SUI @3.4%):${er_nm_sui:12,.2f}")
print(f"TOTAL COMPANY CASH OUTFLOW FOR RUN:    ${total_company_cash_outflow:12,.2f}")
print("=" * 60)
print("      UPDATED SYSTEM YEAR-TO-DATE METRICS (FROM LIVE DB)     ")
print("=" * 60)
print(f"Total Cumulative YTD Gross Salaries:   ${df_history['gross_wages'].sum():12,.2f}")
print(f"Total Cumulative YTD 401(k) Deferred:  ${df_history['pre_tax_401k'].sum():12,.2f}")
print(f"Total Combined Company Overhead Cost:  ${df_history['total_company_cost'].sum():12,.2f}")
print(f"Total Historical Payroll Paychecks Run: {len(df_history)}")
print("=" * 60)


      PAYROLL OVERVIEW FOR: EE-001 | PERIOD: 2026-06-15   
Current Period Gross Wages:            $    5,200.00
Current Pre-Tax 401(k) Allocation:     $      250.00
EMPLOYEE NET TAKE-HOME PAY:            $    3,735.80
------------------------------------------------------------
Employer Matching FICA Social Security: $      322.40
Employer State Unemployment (SUI @3.4%):$      176.80
TOTAL COMPANY CASH OUTFLOW FOR RUN:    $    5,775.45
      UPDATED SYSTEM YEAR-TO-DATE METRICS (FROM LIVE DB)     
Total Cumulative YTD Gross Salaries:   $   15,215.38
Total Cumulative YTD 401(k) Deferred:  $      730.77
Total Combined Company Overhead Cost:  $   16,940.31
Total Historical Payroll Paychecks Run: 3


In [16]:
# --- DATA-CLEANUP & CORRECTION UTILITY ---
import sqlite3
import pandas as pd

DB_NAME = "payroll_ledger.db"
CSV_BACKUP_NAME = "payroll_history_backup.csv"

# 🛑 CONFIGURATION TARGETS: Change these variables to clean up your data
TARGET_PERIOD_DATE = "2026-06-15"   # The pay period date you want to inspect or delete
EXECUTE_DELETION = False            # Change to True ONLY when you are ready to delete
RESET_WHOLE_DATABASE = False        # Change to True ONLY to erase EVERYTHING and start over

# 1. Connect to DB and pull a live overview
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# --- OPTION A: FULL SYSTEM RESET ---
if RESET_WHOLE_DATABASE:
    cursor.execute("DROP TABLE IF EXISTS payroll_ledger")
    conn.commit()
    print("⚠️ WARNING: The 'payroll_ledger' table has been completely wiped from the database.")
    print("Run Cell 1 of your notebook to re-initialize an empty database structure.")

# --- OPTION B: TARGETED PERIOD CLEANUP ---
else:
    # Check if the targeted period exists
    cursor.execute("SELECT id, employee_id, pay_period_ending, gross_wages, net_pay FROM payroll_ledger WHERE pay_period_ending = ?", (TARGET_PERIOD_DATE,))
    records = cursor.fetchall()
    
    if not records:
        print(f"ℹ️ No database records found matching the date: '{TARGET_PERIOD_DATE}'")
    else:
        print(f"🔍 Found {len(records)} entries for the pay period ending '{TARGET_PERIOD_DATE}':")
        for r in records:
            print(f"    -> Record ID: {r[0]} | Employee: {r[1]} | Gross: ${r[3]:,.2f} | Net: ${r[4]:,.2f}")
        
        if EXECUTE_DELETION:
            # Delete targeted rows
            cursor.execute("DELETE FROM payroll_ledger WHERE pay_period_ending = ?", (TARGET_PERIOD_DATE,))
            conn.commit()
            print(f"\n💥 SUCCESS: Deleted all records for pay period '{TARGET_PERIOD_DATE}' from SQLite.")
            
            # Instantly synchronize the changes to your CSV backup file
            df_sync = pd.read_sql_query("SELECT * FROM payroll_ledger", conn)
            df_sync.to_csv(CSV_BACKUP_NAME, index=False)
            print(f"🔄 SUCCESS: Flat ledger CSV file '{CSV_BACKUP_NAME}' has been synchronized.")
        else:
            print("\n🛡️ SAFE MODE ACTIVE: No data was modified.")
            print(f"   To delete the entries listed above, flip 'EXECUTE_DELETION = True' and rerun this cell.")

conn.close()


🔍 Found 3 entries for the pay period ending '2026-06-15':
    -> Record ID: 1 | Employee: EE-001 | Gross: $5,007.69 | Net: $3,328.82
    -> Record ID: 2 | Employee: EE-001 | Gross: $5,007.69 | Net: $3,328.82
    -> Record ID: 3 | Employee: EE-001 | Gross: $5,200.00 | Net: $3,735.80

🛡️ SAFE MODE ACTIVE: No data was modified.
   To delete the entries listed above, flip 'EXECUTE_DELETION = True' and rerun this cell.
